## How to upload files and download files from S3?

In [ ]:
import pandas as pd
import boto3
from dotenv import load_dotenv

In [ ]:
load_dotenv()
bucket_name = 'makeathon-test' # Change this to a unique bucket name!
prefix = "data/"

In [ ]:
# 1. Create dummy data to save to s3 and read from it
data = {
    'name': ['Alice', 'Bob', 'Charlie'],
    'age': [25, 30, 35],
    'email': ['alice@example.com', 'bob@example.com', 'charlie@example.com']
}
df = pd.DataFrame(data)

In [ ]:
# 2. Store file locally
local_file_path = "dummy_data.csv"
df.to_csv(local_file_path, index=False)
print(f"CSV saved locally at: {local_file_path}")

In [ ]:
# 3. Create s3 bucket:
s3 = boto3.resource('s3')
existing_buckets = [bucket.name for bucket in s3.buckets.all()]
s3_client = boto3.client('s3', region_name = "eu-central-1")
if bucket_name not in existing_buckets:
    s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': 'eu-central-1'})
    print("Created new bucket")
else: 
    print("Bucket already exists")

In [ ]:
# 4. Upload to S3
s3_key = prefix + "dummy_data.csv"
s3_client.upload_file(Filename=local_file_path, Bucket=bucket_name, Key=s3_key)
print(f"CSV uploaded to s3://{bucket_name}/{s3_key}")

In [ ]:
# List the content of your folder (prefix)
folder = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix, MaxKeys=100)['Contents']
for file in folder:
    print(file["Key"])

In [ ]:
# Download a file and store it locally
downloaded_file_path = 'downloaded_dummy_data.csv'
s3_client.download_file(Filename=downloaded_file_path, Bucket=bucket_name, Key=s3_key)